In [0]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('maxent_ne_chunker')
nltk.download('words')

In [0]:
%sh
sudo apt-get update -y
sudo apt-get install -y tesseract-ocr
sudo apt-get install -y libtesseract-dev
/databricks/python3/bin/pip install pytesseract

In [0]:
%sh
/databricks/python3/bin/pip install spacy 
/databricks/python3/bin/python3 -m spacy download en_core_web_sm
# /databricks/python3/bin/pip install pytesseract

In [0]:
%pip install opencv-python
# %pip install pytesseract

In [0]:
dbutils.library.restartPython()
# import sys
# print(sys.version)
# !pip install en_core_web_sm

In [0]:
import pytesseract
import cv2
import matplotlib.pyplot as plt
import PyPDF2
from PyPDF2 import PdfReader
import os

import spacy 
nlp = spacy.load("en_core_web_sm")
from spacy import displacy

import re
import nltk
from nltk import sent_tokenize, word_tokenize, pos_tag

Reading text from scanned document 

In [0]:
path2 = r"Imagepath" 

In [0]:
image = cv2.imread(path2)
processing_image = image.copy()
base_image = image.copy()

In [0]:

def display(im_data):
    dpi = 80
    max_height = 1000
    max_width = 1000
    if im_data.shape[0] > max_height or im_data.shape[1] > max_width:
        scale = min(max_height / im_data.shape[0], max_width / im_data.shape[1])
        im_data = cv2.resize(im_data, (0, 0), fx=scale, fy=scale)
    height, width = im_data.shape[:2]
    figsize = width / float(dpi), height / float(dpi)
    fig = plt.figure(figsize=figsize)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis('off')
    ax.imshow(im_data, cmap='gray')
    plt.show()


In [0]:
display(image)

Image Preprocessing: 

In [0]:
def image_dilation(processing_image):
    gray = cv2.cvtColor(processing_image, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (7,7), 0)
    thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    kernal = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 50))
    dilate = cv2.dilate(thresh, kernal, iterations=1)
    # cv2.imwrite("temp/sample_dilate.png", dilate)
    cnts = cv2.findContours(dilate, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnts = cnts[0] if len(cnts) == 2 else cnts[1]
    cnts = sorted(cnts, key=lambda x: cv2.boundingRect(x)[1])
    for c in cnts:
        x,y,w,h = cv2.boundingRect(c)
        if h > 200 and w > 250:
            roi = base_image[y:y+h, x:x+w]
            # print(roi)
            cv2.rectangle(processing_image, (x,y), (x+w, y+h), (36, 255, 12), 2)
    return (processing_image)

In [0]:
new_image=image_dilation(processing_image)

In [0]:
display (new_image)

In [0]:
# ocr_result_new = pytesseract.image_to_string(roi)
ocr_result_new = pytesseract.image_to_string(new_image)
print(ocr_result_new)

In [0]:
ocr_result_original = pytesseract.image_to_string(base_image)
print(ocr_result_original)

Using NER to find PII from extracted text 

In [0]:
doc = nlp(ocr_result_new)
doc.ents

In [0]:
displacy.render(doc, style="ent", jupyter=True)

PII Extraction from PDF Folder:


In [0]:
def extract_text_from_pdf(pdf_path):
    with open(pdf_path, 'rb') as file:
        reader = PdfReader(file)
        text = ''
        for page in reader.pages:
            text += page.extract_text()
    return text



In [0]:
def process_text_spacy(text):
    doc = nlp(text)
    names = [ent.text for ent in doc.ents if ent.label_ == 'PERSON']
    document_ids= (re.findall(r'[A-Z]\d{3}-\d{5}-\d{5}', text))
    return names, document_ids

def calculate_entity_probabilities_spacy(pdf_folder):
    probabilities = {}
    for file_name in os.listdir(pdf_folder):
        if file_name.endswith('.pdf'):
            file_path = os.path.join(pdf_folder, file_name)
            text = extract_text_from_pdf(file_path)
            entities = process_text_spacy(text)
            # print(entities)
            total_words = len(text.split())
            entity_count = sum(len(sublist) for sublist in entities)
            entity_probability = entity_count / total_words if total_words > 0 else 0
            probabilities[file_name] = [entity_count,entity_probability]
    return probabilities

In [0]:
def process_text_nltk(text):
    sentences = nltk.sent_tokenize(text)
    names = []
    document_ids = []
    for sentence in sentences:
        words = nltk.word_tokenize(sentence)
        tagged_words = nltk.pos_tag(words)
        named_entities = nltk.ne_chunk(tagged_words)
        for entity in named_entities:
            if isinstance(entity, nltk.Tree):
                if entity.label() == 'PERSON':
                    names.append(' '.join([leaf[0] for leaf in entity.leaves()]))
        document_ids.extend(re.findall(r'[A-Z]\d{3}-\d{5}-\d{5}', sentence))
    return names, document_ids


def calculate_entity_probabilities_nltk(pdf_folder):
    probabilities = {}
    for file_name in os.listdir(pdf_folder):
        if file_name.endswith('.pdf'):
            file_path = os.path.join(pdf_folder, file_name)
            text = extract_text_from_pdf(file_path)
            entities = process_text_nltk(text)
            # print(entities)
            total_words = len(text.split())
            entity_count = sum(len(sublist) for sublist in entities)
            entity_probability = entity_count / total_words if total_words > 0 else 0
            probabilities[file_name] = [entity_count,entity_probability]
    return probabilities



In [0]:

text = "In a bustling city like Nidhi table, Toronto, Sarah Johnson and John Smith often find themselves navigating the busy streets. Meanwhile, Emily Brown and Michael Davis prefer the serene countryside of Ontario. As they embark on their daily commutes, they ensure to have their Canadian driving license numbers handy. Sarah carries her license with number A868-56789-12345, while John's license bears the number A868-54321-98765. Similarly, Emily's license is registered with the number A868-24680-13579, and Michael's license number is A8768-97531-24680. With their licenses securely tucked away in their wallets, they confidently set out on their journeys, knowing they're prepared for whatever the road may bring."
Result = process_text_nltk(text)
print(Result)

In [0]:
text = "In a bustling city like Nidhi Toronto, Sarah Johnson and John Smith often find themselves navigating the busy streets. Meanwhile, Emily Brown and Michael Davis prefer the serene countryside of Ontario. As they embark on their daily commutes, they ensure to have their Canadian driving license numbers handy. Sarah carries her license with number A868-56789-12345, while John's license bears the number A868-5431-98765. Similarly, Emily's license is registered with the number A868-24680-13579, and Michael's license number is A868-97531-24680. With their licenses securely tucked away in their wallets, they confidently set out on their journeys, knowing they're prepared for whatever the road may bring."
Result = process_text_spacy(text)
print(Result)

In [0]:
pdf_folder = 'Pdf_folder_path'
entity_probabilities = calculate_entity_probabilities_spacy(pdf_folder)
for file_name, data in entity_probabilities.items():
    PII_Count = data[0]
    probability = data[1]
    print(f"File: {file_name}, PII_Count: {PII_Count}, Probability: {probability:.2%}")

In [0]:

pdf_folder = 'Pdf_folder_path'
entity_probabilities = calculate_entity_probabilities_nltk(pdf_folder)
for file_name, data in entity_probabilities.items():
    PII_Count = data[0]
    print(f"File: {file_name}, PII_Count: {PII_Count}, Probability: {probability:.2%}")

Redact PII from PDF:

In [0]:
def redact_pii(text, pii_list):
    redacted_text = text
    for pii in pii_list:
        redacted_text = re.sub(r'\b{}\b'.format(re.escape(pii)), '[REDA######]', redacted_text)
    return redacted_text



In [0]:
pdf_path = 'path_240003010.pdf'
text = extract_text_from_pdf(pdf_path)
names, document_ids = process_text_spacy(text)

pii_list = names + document_ids
redacted_text = redact_pii(text, pii_list)

print(redacted_text)

In [0]:
# Saving redacted text in form of new pdf 

from PyPDF2 import PdfReader, PdfWriter

def return_redacted_pdf(redacted_text, pdf_path, output_path):
    with open(output_path, 'wb') as file:
        writer = PdfWriter()
        for page in PdfReader(pdf_path).pages:
            writer.add_page(page)
        writer.write(file)
        

In [0]:
redacted_pdf_path = 'path_4054003010_redacted.pdf'
input_pdf_path = 'path_054003010.pdf'
return_redacted_pdf(redacted_text,input_pdf_path, redacted_pdf_path)